In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
magic_gamma_telescope = fetch_ucirepo(id=159) 
  
# data (as pandas dataframes) 
X = magic_gamma_telescope.data.features 
y = magic_gamma_telescope.data.targets 
  
# metadata 
print(magic_gamma_telescope.metadata) 
  
# variable information 
print(magic_gamma_telescope.variables) 

In [ ]:
import tensorflow as tf
from tensorflow import keras
import sklearn
from keras import layers


In [ ]:
# Capa personalizada que implementa una combinación de polinomios de Legendre
class PolynomialDense(tf.keras.layers.Layer):
    def __init__(self, units, degree=2, use_bias=True, **kwargs):
        super(PolynomialDense, self).__init__(**kwargs)
        self.units = units
        self.degree = degree
        self.use_bias = use_bias

    def build(self, input_shape):
        input_dim = input_shape[-1]

        self.kernels = []
        for d in range(1, self.degree + 1):
            w = self.add_weight(
                shape=(input_dim, self.units),
                initializer='glorot_uniform',
                trainable=True,
                name=f'kernel_degree_{d}'
            )
            self.kernels.append(w)

        if self.use_bias:
            self.bias = self.add_weight(
                shape=(self.units,),
                initializer='zeros',
                trainable=True,
                name='bias'
            )

    def call(self, inputs):

        #Asumimos que los inputs ya están normalizados en [-1, 1]
        x = inputs

        # --- 2. Generamos la base de Legendre
        # P0(x)
        P0 = tf.ones_like(x)

        if self.degree == 0:
            features = [P0]
        else:
            # P1(x)
            P1 = x
            features = [P1]

            if self.degree > 1:
                Pnm2 = P0
                Pnm1 = P1

                for n in range(2, self.degree + 1):
                    Pn = ((2.0 * n - 1.0) * x * Pnm1 - (n - 1.0) * Pnm2) / n
                    features.append(Pn)

                    Pnm2 = Pnm1
                    Pnm1 = Pn

        # --- 3. Combinación lineal con los kernels
        output = 0.0

        for phi, W in zip(features, self.kernels):
            output += tf.matmul(phi, W)

        if self.use_bias:
            output += self.bias

        return output


In [ ]:
class PolynomialDense2(tf.keras.layers.Layer):
    def __init__(self, units, degree=2, use_bias=True, **kwargs):
        super(PolynomialDense2, self).__init__(**kwargs)
        self.units = units
        self.degree = degree
        self.use_bias = use_bias

    def build(self, input_shape):
        input_dim = input_shape[-1]
        # El kernel debe cubrir (grado + 1) si incluyes P0, o (grado) si empiezas en P1
        # Aquí usaremos desde P1 hasta P_degree
        self.kernel = self.add_weight(
            shape=(input_dim * self.degree, self.units),
            initializer=tf.keras.initializers.GlorotUniform(),
            trainable=True,
            name="kernel"
        )

        if self.use_bias:
            self.bias = self.add_weight(
                shape=(self.units,),
                initializer="zeros",
                trainable=True,
                name="bias"
            )

    def call(self, inputs):
        # Aseguramos que los inputs sean float32
        x = tf.cast(inputs, self.compute_dtype)
        
        # P0 = 1, P1 = x
        p_nm2 = tf.ones_like(x)
        p_nm1 = x
        
        features = [p_nm1] # Empezamos con grado 1

        for n in range(2, self.degree + 1):
            n_float = tf.cast(n, self.compute_dtype)
            # Fórmula de recurrencia de Legendre
            p_n = ((2.0 * n_float - 1.0) * x * p_nm1 - (n_float - 1.0) * p_nm2) / n_float
            features.append(p_n)
            
            p_nm2 = p_nm1
            p_nm1 = p_n

        # Concatenamos: (batch, input_dim * degree)
        # Esto genera: [x_poly1, x_poly2, ..., x_poly_degree]
        phi = tf.concat(features, axis=-1)

        output = tf.matmul(phi, self.kernel)

        if self.use_bias:
            output = tf.nn.bias_add(output, self.bias)

        return output

In [ ]:
#Dividimos el dataset en entrenamiento y prueba
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size=0.3, random_state=1)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
#Normalizamos los datos
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1, 1))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
#Cambiamos las etiquetas a 0 y 1
y_train = (y_train == 'g').astype(int)
y_test = (y_test == 'g').astype(int)

In [ ]:
input_dim = X_train.shape[1]

#Modelo lineal
inputLinear = keras.Input(shape=(input_dim,))
x1 = layers.Dense(32, activation='relu')(inputLinear)
x2 = layers.Dense(32, activation='relu')(x1)
outputLinear = layers.Dense(3, activation='softmax')(x2)

#MODELO DE RED POLINÓMICA DE LEGENDRE

#Modelo polinómico grado 2

inputPoli = keras.Input(shape=(input_dim,))

x = PolynomialDense2(32, degree=2)(inputPoli)
x = layers.Activation('tanh')(x)
x = layers.Dense(32, activation='tanh')(x)

outputPoli = layers.Dense(3, activation='softmax')(x)

#Modelo polinómico grado 3
inputPoli3 = keras.Input(shape=(input_dim,))
x3 = PolynomialDense2(32, degree=3)(inputPoli3)
x3 = layers.Activation('tanh')(x3)
x3 = layers.Dense(32, activation='tanh')(x3)
outputPoli3 = layers.Dense(3, activation='softmax')(x3)

#Modelo polinómico grado 4
inputPoli4 = keras.Input(shape=(input_dim,))
x4 = PolynomialDense2(32, degree=4)(inputPoli4)
x4 = layers.Activation('tanh')(x4)
x4 = layers.Dense(32, activation='tanh')(x4)
outputPoli4 = layers.Dense(3, activation='softmax')(x4)

In [ ]:
#Guardamos ambos modelos
modelLinear  = keras.Model(inputs=inputLinear, outputs=outputLinear)
modelPoli = keras.Model(inputs=inputPoli, outputs=outputPoli) 
modelPoli3 = keras.Model(inputs=inputPoli3, outputs=outputPoli3)
modelPoli4 = keras.Model(inputs=inputPoli4, outputs=outputPoli4)



In [ ]:
modelPoli3.summary()

In [ ]:
#Compilamos ambos modelos
modelLinear.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelPoli.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelPoli3.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelPoli4.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
epochs = 50

In [ ]:
#Añadimos Early Stopping
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
#Entrenamos el modelo
historyLinear = modelLinear.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])
historyPoli = modelPoli.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])
historyPoli3 = modelPoli3.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])
historyPoli4 = modelPoli4.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])

In [ ]:
#Comparamos los resultados de ambos modelos
lossLinear, accLinear = modelLinear.evaluate(X_test, y_test, verbose=0)
lossPoli, accPoli = modelPoli.evaluate(X_test, y_test, verbose=0)
lossPoli3, accPoli3 = modelPoli3.evaluate(X_test, y_test, verbose=0)
lossPoli4, accPoli4 = modelPoli4.evaluate(X_test, y_test, verbose=0)

print(f'Modelo Lineal - Loss: {lossLinear:.4f}, Accuracy: {accLinear:.4f}')
print(f'Modelo Polinómico Grado 2 - Loss: {lossPoli:.4f}, Accuracy: {accPoli:.4f}')
print(f'Modelo Polinómico Grado 3 - Loss: {lossPoli3:.4f}, Accuracy: {accPoli3:.4f}')
print(f'Modelo Polinómico Grado 4 - Loss: {lossPoli4:.4f}, Accuracy: {accPoli4:.4f}')

In [ ]:
#creamos una funcion para crear un modelo lineal y un modelo polinómico de Legendre

def build_linear_model():
    inputLinear = keras.Input(shape=(input_dim,))
    x1 = layers.Dense(32, activation='relu')(inputLinear)
    x2 = layers.Dense(32, activation='relu')(x1)
    outputLinear = layers.Dense(3, activation='softmax')(x2)
    modelLinear = keras.Model(inputs=inputLinear, outputs=outputLinear)
    modelLinear.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return modelLinear

def build_poly_model(degree):
    inputPoli = keras.Input(shape=(input_dim,))
    x = PolynomialDense2(32, degree=degree)(inputPoli)
    x = layers.Activation('tanh')(x)
    x = layers.Dense(16, activation='relu')(x)
    outputPoli = layers.Dense(3, activation='softmax')(x)
    modelPoli = keras.Model(inputs=inputPoli, outputs=outputPoli)
    modelPoli.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return modelPoli

# Cross Validation.

In [ ]:
import numpy as np
from tqdm import tqdm

In [ ]:
X = magic_gamma_telescope.data.features 
y = magic_gamma_telescope.data.targets 

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

scoreLinear = []
scorePoli = []
scorePoli3 = []
scorePoli4 = []
X = X.to_numpy()
y = y.to_numpy()



for train_index, test_index in tqdm(skf.split(X, y), total=10):

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Normalización
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # etiquetas binarias
    y_train = (y_train == 'g').astype(int)
    y_test = (y_test == 'g').astype(int)

    #CREAR MODELOS NUEVOS EN CADA FOLD
    modelLinear = build_linear_model()
    modelPoli = build_poly_model(2)
    modelPoli3 = build_poly_model(3)
    modelPoli4 = build_poly_model(4)

    # Early stopping nuevo
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    # Entrenamiento
    modelLinear.fit(X_train, y_train, epochs=epochs, batch_size=16,
                    validation_data=(X_test, y_test),
                    callbacks=[early_stopping],
                    verbose=0)

    modelPoli.fit(X_train, y_train, epochs=epochs, batch_size=16,
                  validation_data=(X_test, y_test),
                  callbacks=[early_stopping],
                  verbose=0)

    modelPoli3.fit(X_train, y_train, epochs=epochs, batch_size=16,
                   validation_data=(X_test, y_test),
                   callbacks=[early_stopping],
                   verbose=0)

    modelPoli4.fit(X_train, y_train, epochs=epochs, batch_size=16,
                   validation_data=(X_test, y_test),
                   callbacks=[early_stopping],
                   verbose=0)

    # Evaluación
    _, accLinear = modelLinear.evaluate(X_test, y_test, verbose=0)
    _, accPoli = modelPoli.evaluate(X_test, y_test, verbose=0)
    _, accPoli3 = modelPoli3.evaluate(X_test, y_test, verbose=0)
    _, accPoli4 = modelPoli4.evaluate(X_test, y_test, verbose=0)

    scoreLinear.append(accLinear)
    scorePoli.append(accPoli)
    scorePoli3.append(accPoli3)
    scorePoli4.append(accPoli4)

In [ ]:
#imprimimos los resultados
print(f'Modelo Lineal - Accuracy: {np.mean(scoreLinear):.4f}' ' ± ' + f'{np.std(scoreLinear):.4f}')
print(f'Modelo Polinómico - Accuracy: {np.mean(scorePoli):.4f}' ' ± ' + f'{np.std(scorePoli):.4f}')
print(f'Modelo Polinómico 3 - Accuracy: {np.mean(scorePoli3):.4f}' ' ± ' + f'{np.std(scorePoli3):.4f}')
print(f'Modelo Polinómico 4 - Accuracy: {np.mean(scorePoli4):.4f}' ' ± ' + f'{np.std(scorePoli4):.4f}')